
# cap10_foldlocal_multi_objective_readiness_v1

Objective: audit whether the latest cap10 handoff can safely enter another training loop without leaking labels or repeating an already failed overfit path.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison / current handoff: `cap10_training_objective_handoff_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection`, `training_objective`, `checkpoint_selection`.
- Target window: fold-local `wf_2022` and `wf_2023_2024` evidence; next full run must restore `wf_2020` coverage before promotion.
- Acceptance gate: minimum relative return +30pp, maximum drawdown <= 15%, fold-local target scope, no eval overlap, and no runtime promotion from target readiness only.
- External basis: Qlib artifact/replay discipline, PBO/CSCV, Deflated Sharpe, purged/embargo validation, conservative offline RL, and risk-sensitive RL.

Closest historical success: `sparse_rebound_release_target_revision_v1` produced fold-local sparse-release labels with scope/lift/readiness gates.

Closest historical failure: `sparse_release_bc_ppo_screen_v1` and `sparse_release_drawdown_control_bc_ppo_screen_v1` did not clear fold gates; PPO selection worsened drawdown on `wf_2022`.

Why this is not a duplicate: this notebook does not tune thresholds or rerun PPO. It audits the cap10 handoff against existing target tables and post-target training evidence, then writes an anti-overfit stop-rule contract for the next training attempt.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)

EXPERIMENT = "cap10_foldlocal_multi_objective_readiness_v1"
ROOT = Path.cwd()
CARD_ROOT = ROOT / "runtime" / "model_selection" / "card"
OUT_DIR = CARD_ROOT / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0


In [2]:

def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def bool_col(frame: pd.DataFrame, col: str) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(False, index=frame.index)
    values = frame[col]
    if values.dtype == bool:
        return values.fillna(False)
    return values.fillna(False).astype(str).str.lower().isin(["true", "1", "yes"])


def numeric_col(frame: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(default, index=frame.index, dtype="float64")
    return pd.to_numeric(frame[col], errors="coerce").fillna(default)


def derive_folds_from_targets(targets: pd.DataFrame) -> tuple[list[dict[str, str]], pd.DataFrame]:
    required = {"fold", "train_start", "train_end", "eval_start", "eval_end"}
    if not required.issubset(targets.columns):
        return [], pd.DataFrame()
    meta = targets[list(required)].dropna().copy()
    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        meta[col] = pd.to_datetime(meta[col]).dt.strftime("%Y-%m-%d")
    boundary_rows = []
    folds = []
    for fold, group in meta.groupby("fold", sort=True):
        unique = group.drop_duplicates().reset_index(drop=True)
        boundary_rows.append(
            {
                "fold": str(fold),
                "boundary_variant_count": int(len(unique)),
                "train_start_values": ",".join(sorted(unique["train_start"].unique())),
                "train_end_values": ",".join(sorted(unique["train_end"].unique())),
                "eval_start_values": ",".join(sorted(unique["eval_start"].unique())),
                "eval_end_values": ",".join(sorted(unique["eval_end"].unique())),
            }
        )
        first = unique.iloc[0]
        folds.append(
            {
                "name": str(fold),
                "train_start": str(first["train_start"]),
                "train_end": str(first["train_end"]),
                "eval_start": str(first["eval_start"]),
                "eval_end": str(first["eval_end"]),
            }
        )
    return folds, pd.DataFrame(boundary_rows)


def effective_target_position(targets: pd.DataFrame) -> pd.Series:
    teacher = numeric_col(targets, "teacher_position")
    rebound = numeric_col(targets, "rebound_target_position", default=np.nan).fillna(teacher)
    risk = numeric_col(targets, "risk_target_position", default=np.nan).fillna(teacher)
    is_rebound = bool_col(targets, "is_rebound_cluster")
    is_downtrend = bool_col(targets, "is_downtrend_state")
    return pd.Series(
        np.where(is_downtrend, risk, np.where(is_rebound, rebound, teacher)),
        index=targets.index,
    ).clip(0.0, 1.0)


def action_distribution(targets: pd.DataFrame) -> dict[str, Any]:
    if targets.empty:
        return {"samples": 0, "unique_actions": 0, "dominant_action_share": 1.0, "action_entropy": 0.0}
    actions = (effective_target_position(targets) / 0.05).round().astype(int).clip(0, 20)
    counts = actions.value_counts(normalize=True)
    entropy = float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0
    return {
        "samples": int(len(actions)),
        "unique_actions": int(actions.nunique()),
        "dominant_action_share": float(counts.iloc[0]) if not counts.empty else 1.0,
        "action_entropy": entropy,
        "dominant_action": int(counts.index[0]) if not counts.empty else -1,
    }


def target_asset_summary(name: str, rel_path: str, source_selection: dict[str, Any], post_training_note: str) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    path = ROOT / rel_path
    targets = read_csv(path)
    row: dict[str, Any] = {
        "asset": name,
        "path": rel_path,
        "exists": bool(path.exists()),
        "source_decision": source_selection.get("decision", ""),
        "source_training_allowed": bool(source_selection.get("training_allowed", False) or source_selection.get("bc_training_allowed_next", False)),
        "source_promotion_allowed": bool(source_selection.get("promotion_allowed", False)),
        "post_training_note": post_training_note,
    }
    if targets.empty:
        row.update({"rows": 0, "ready_for_cap10_as_is": False, "blocker": "missing_or_empty_target_table"})
        return row, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if "risk_target_position_original" not in targets.columns and "risk_target_position" in targets.columns:
        targets["risk_target_position_original"] = targets["risk_target_position"]
    for col in ["is_rebound_cluster", "is_downtrend_state"]:
        if col in targets.columns:
            targets[col] = bool_col(targets, col)

    folds, fold_boundary_audit = derive_folds_from_targets(targets)
    if folds:
        scope_audit = audit_state_objective_target_scope(targets, folds=folds)
        scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
    else:
        scope_audit = pd.DataFrame()
        scope_summary = {"training_allowed": False, "eval_overlap_rows": -1, "scope_issues": ["missing_fold_boundaries"]}

    lift_required = {"teacher_position", "rebound_target_position", "risk_target_position", "is_rebound_cluster", "is_downtrend_state"}
    if lift_required.issubset(targets.columns):
        lift_audit = audit_state_objective_target_lift(targets, baseline_risk_position_col="risk_target_position_original")
        lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
    else:
        lift_audit = pd.DataFrame()
        lift_summary = {
            "training_allowed": False,
            "rebound_rows": int(bool_col(targets, "is_rebound_cluster").sum()),
            "current_positive_rebound_lift_rows": 0,
            "candidate_release_positive_lift_rows": 0,
            "rebound_downtrend_rows": 0,
            "next_mechanism": "add_required_target_columns",
        }

    dist = action_distribution(targets)
    bc_readiness_gate = bool(dist["samples"] >= 100 and dist["unique_actions"] >= 3 and dist["dominant_action_share"] <= 0.90)
    rebound_rows = int(bool_col(targets, "is_rebound_cluster").sum())
    downtrend_rows = int(bool_col(targets, "is_downtrend_state").sum())
    current_lift_rows = int(lift_summary.get("current_positive_rebound_lift_rows", 0))
    scope_gate = bool(scope_summary.get("training_allowed", False))
    boundary_gate = bool(fold_boundary_audit.empty or int(fold_boundary_audit["boundary_variant_count"].max()) == 1)
    multi_objective_gate = bool(rebound_rows > 0 and downtrend_rows > 0 and current_lift_rows > 0)

    prior_failed_as_is = any(token in post_training_note.lower() for token in ["failed", "mixed", "weak", "conflict"])
    ready_for_cap10_as_is = bool(scope_gate and boundary_gate and multi_objective_gate and bc_readiness_gate and not prior_failed_as_is)
    blockers = []
    if not scope_gate:
        blockers.append("scope")
    if not boundary_gate:
        blockers.append("fold_boundary_inconsistent")
    if not multi_objective_gate:
        blockers.append("missing_rebound_lift_or_drawdown_state")
    if not bc_readiness_gate:
        blockers.append("bc_label_distribution")
    if prior_failed_as_is:
        blockers.append("prior_post_target_screen_failed_or_mixed")

    row.update(
        {
            "rows": int(len(targets)),
            "folds": ",".join(sorted(str(x) for x in targets.get("fold", pd.Series(dtype=str)).dropna().unique())),
            "target_scope_values": ",".join(sorted(str(x) for x in targets.get("target_scope", pd.Series(dtype=str)).dropna().unique())),
            "scope_gate": scope_gate,
            "eval_overlap_rows": int(scope_summary.get("eval_overlap_rows", 0)),
            "boundary_gate": boundary_gate,
            "max_boundary_variant_count": int(fold_boundary_audit["boundary_variant_count"].max()) if not fold_boundary_audit.empty else 0,
            "rebound_rows": rebound_rows,
            "downtrend_rows": downtrend_rows,
            "rebound_downtrend_rows": int(lift_summary.get("rebound_downtrend_rows", 0)),
            "current_positive_rebound_lift_rows": current_lift_rows,
            "candidate_release_positive_lift_rows": int(lift_summary.get("candidate_release_positive_lift_rows", 0)),
            "target_lift_gate": bool(lift_summary.get("training_allowed", False)),
            "multi_objective_gate": multi_objective_gate,
            "bc_readiness_gate": bc_readiness_gate,
            **{f"bc_{k}": v for k, v in dist.items()},
            "ready_for_cap10_as_is": ready_for_cap10_as_is,
            "blocker": ",".join(blockers),
        }
    )
    return row, scope_audit, lift_audit, fold_boundary_audit


In [3]:

source_selections = {
    name: load_json(CARD_ROOT / name / "selection.json")
    for name in [
        "cap10_training_objective_handoff_v1",
        "fold_local_joint_reward_targets_v1",
        "fold_local_rebound_priority_conflict_v1",
        "sparse_rebound_release_target_revision_v1",
        "sparse_release_bc_ppo_screen_v1",
        "sparse_release_drawdown_control_foldlocal_bc_warmstart_v1",
        "sparse_release_drawdown_control_bc_ppo_screen_v1",
        "sparse_release_drawdown_control_checkpoint_score_v1",
    ]
}

assets = [
    (
        "fold_local_joint_reward_targets_v1",
        "runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv",
        source_selections["fold_local_joint_reward_targets_v1"],
        "blocked later by fold_local_rebound_priority_conflict_v1: current positive rebound lift was erased by risk-priority target semantics",
    ),
    (
        "sparse_rebound_release_target_revision_v1",
        "runtime/model_selection/card/sparse_rebound_release_target_revision_v1/sparse_release_revised_targets.csv",
        source_selections["sparse_rebound_release_target_revision_v1"],
        "mixed later in sparse_release_bc_ppo_screen_v1: one fold passed but wf_2022 selected PPO drifted into high drawdown",
    ),
    (
        "sparse_release_drawdown_control_foldlocal_bc_warmstart_v1",
        "runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1/selected_foldlocal_targets.csv",
        source_selections["sparse_release_drawdown_control_foldlocal_bc_warmstart_v1"],
        "failed later in sparse_release_drawdown_control_bc_ppo_screen_v1: drawdown-control-only target did not clear fold gates",
    ),
]

summary_rows = []
scope_audits = []
lift_audits = []
boundary_audits = []
for asset_name, rel_path, selection, note in assets:
    row, scope, lift, boundary = target_asset_summary(asset_name, rel_path, selection, note)
    summary_rows.append(row)
    if not scope.empty:
        scope.insert(0, "asset", asset_name)
        scope_audits.append(scope)
    if not lift.empty:
        lift.insert(0, "asset", asset_name)
        lift_audits.append(lift)
    if not boundary.empty:
        boundary.insert(0, "asset", asset_name)
        boundary_audits.append(boundary)

target_asset_readiness = pd.DataFrame(summary_rows)
target_scope_audit = pd.concat(scope_audits, ignore_index=True) if scope_audits else pd.DataFrame()
target_lift_audit = pd.concat(lift_audits, ignore_index=True) if lift_audits else pd.DataFrame()
fold_boundary_audit = pd.concat(boundary_audits, ignore_index=True) if boundary_audits else pd.DataFrame()

target_asset_readiness


,asset,path,exists,source_decision,source_training_allowed,source_promotion_allowed,post_training_note,rows,folds,target_scope_values,...,target_lift_gate,multi_objective_gate,bc_readiness_gate,bc_samples,bc_unique_actions,bc_dominant_action_share,bc_action_entropy,bc_dominant_action,ready_for_cap10_as_is,blocker
0,fold_local_joint_reward_targets_v1,runtime/model_selection/card/fold_local_joint_...,True,fold_local_joint_reward_targets_ready_for_weig...,True,False,blocked later by fold_local_rebound_priority_c...,1261,"wf_2022_train,wf_2023_2024_train",fold_local,...,False,False,True,1261,18,0.221253,2.347699e+00,4,False,"missing_rebound_lift_or_drawdown_state,prior_p..."
1,sparse_rebound_release_target_revision_v1,runtime/model_selection/card/sparse_rebound_re...,True,sparse_release_targets_bc_screen_ready_no_trai...,True,False,mixed later in sparse_release_bc_ppo_screen_v1...,1261,"wf_2022_train,wf_2023_2024_train",fold_local,...,True,True,True,1261,18,0.221253,2.458781e+00,4,False,prior_post_target_screen_failed_or_mixed
2,sparse_release_drawdown_control_foldlocal_bc_w...,runtime/model_selection/card/sparse_release_dr...,True,foldlocal_bc_warmstart_ready_no_promotion,True,False,failed later in sparse_release_drawdown_contro...,370,wf_2022_train,fold_local,...,False,False,False,370,1,1.000000,-1.000089e-12,5,False,"missing_rebound_lift_or_drawdown_state,bc_labe..."


In [4]:

def load_training_screen(label: str, rel_path: str) -> pd.DataFrame:
    frame = read_csv(ROOT / rel_path)
    if frame.empty:
        return frame
    frame = frame.copy()
    frame.insert(0, "screen", label)
    for prefix in ["bc_test", "selected_test"]:
        rel_col = f"{prefix}_relative_return"
        dd_col = f"{prefix}_max_dd"
        if rel_col in frame.columns and dd_col in frame.columns:
            frame[f"{prefix}_full_gate"] = (pd.to_numeric(frame[rel_col], errors="coerce") >= TARGET_RELATIVE_RETURN) & (
                pd.to_numeric(frame[dd_col], errors="coerce") <= TARGET_MAX_DD
            )
            frame[f"{prefix}_relative_gap"] = (TARGET_RELATIVE_RETURN - pd.to_numeric(frame[rel_col], errors="coerce")).clip(lower=0.0)
            frame[f"{prefix}_drawdown_gap"] = (pd.to_numeric(frame[dd_col], errors="coerce") - TARGET_MAX_DD).clip(lower=0.0)
    if {"selected_test_max_dd", "bc_test_max_dd"}.issubset(frame.columns):
        frame["selected_vs_bc_drawdown_delta"] = pd.to_numeric(frame["selected_test_max_dd"], errors="coerce") - pd.to_numeric(frame["bc_test_max_dd"], errors="coerce")
    if {"selected_test_relative_return", "bc_test_relative_return"}.issubset(frame.columns):
        frame["selected_vs_bc_relative_delta"] = pd.to_numeric(frame["selected_test_relative_return"], errors="coerce") - pd.to_numeric(frame["bc_test_relative_return"], errors="coerce")
    frame["overfit_stop_flag"] = False
    if "selected_vs_bc_drawdown_delta" in frame.columns:
        frame["overfit_stop_flag"] = frame["overfit_stop_flag"] | (frame["selected_vs_bc_drawdown_delta"] > 2.0)
    if "selected_test_full_gate" in frame.columns:
        frame["overfit_stop_flag"] = frame["overfit_stop_flag"] | (~frame["selected_test_full_gate"].fillna(False))
    return frame

training_screens = pd.concat(
    [
        load_training_screen(
            "sparse_release_bc_ppo_screen_v1",
            "runtime/model_selection/card/sparse_release_bc_ppo_screen_v1/ppo_fold_results.csv",
        ),
        load_training_screen(
            "sparse_release_drawdown_control_bc_ppo_screen_v1",
            "runtime/model_selection/card/sparse_release_drawdown_control_bc_ppo_screen_v1/ppo_fold_results.csv",
        ),
    ],
    ignore_index=True,
)

cols = [
    c for c in [
        "screen",
        "fold",
        "bc_test_relative_return",
        "bc_test_max_dd",
        "bc_test_full_gate",
        "selected_test_relative_return",
        "selected_test_max_dd",
        "selected_test_full_gate",
        "selected_vs_bc_relative_delta",
        "selected_vs_bc_drawdown_delta",
        "overfit_stop_flag",
        "bc_gate",
        "selected_gate",
        "fold_gate",
    ] if c in training_screens.columns
]
training_screen_summary = training_screens[cols].copy() if not training_screens.empty else pd.DataFrame(columns=cols)
training_screen_summary


,screen,fold,bc_test_relative_return,bc_test_max_dd,bc_test_full_gate,selected_test_relative_return,selected_test_max_dd,selected_test_full_gate,selected_vs_bc_relative_delta,selected_vs_bc_drawdown_delta,overfit_stop_flag,bc_gate,selected_gate,fold_gate
0,sparse_release_bc_ppo_screen_v1,wf_2022,36.419124,18.455088,False,41.770333,37.275221,False,5.351209,18.820132,True,False,False,False
1,sparse_release_bc_ppo_screen_v1,wf_2023_2024,30.501116,13.597475,True,30.501116,13.597475,True,0.000000,0.000000,False,True,True,True
2,sparse_release_drawdown_control_bc_ppo_screen_v1,wf_2022,54.403539,29.365633,False,46.438325,34.604542,False,-7.965214,5.238909,True,False,False,False
3,sparse_release_drawdown_control_bc_ppo_screen_v1,wf_2023_2024,41.093405,21.460221,False,41.093405,21.460221,False,0.000000,0.000000,True,False,False,False


In [5]:

readiness_gate_count = int(target_asset_readiness["ready_for_cap10_as_is"].sum()) if not target_asset_readiness.empty else 0
best_asset = ""
if readiness_gate_count > 0:
    best_asset = str(target_asset_readiness[target_asset_readiness["ready_for_cap10_as_is"]].iloc[0]["asset"])
else:
    ranked = target_asset_readiness.copy()
    for col in ["scope_gate", "multi_objective_gate", "bc_readiness_gate"]:
        ranked[col] = ranked[col].fillna(False).astype(bool)
    ranked["diagnostic_score"] = ranked[["scope_gate", "multi_objective_gate", "bc_readiness_gate"]].sum(axis=1)
    ranked["diagnostic_score"] += np.where(ranked["current_positive_rebound_lift_rows"] > 0, 1, 0)
    ranked["diagnostic_score"] += np.where(ranked["downtrend_rows"] > 0, 1, 0)
    best_asset = str(ranked.sort_values(["diagnostic_score", "rows"], ascending=[False, False]).iloc[0]["asset"])

selected_training_failures = []
if not training_screen_summary.empty:
    failed = training_screen_summary[training_screen_summary.get("overfit_stop_flag", False).astype(bool)]
    for record in failed.to_dict("records"):
        selected_training_failures.append(
            {
                "screen": record.get("screen", ""),
                "fold": record.get("fold", ""),
                "selected_test_relative_return": record.get("selected_test_relative_return"),
                "selected_test_max_dd": record.get("selected_test_max_dd"),
                "selected_vs_bc_drawdown_delta": record.get("selected_vs_bc_drawdown_delta"),
            }
        )

anti_overfit_stop_rules = pd.DataFrame(
    [
        {
            "rule": "fold_gate_absolute",
            "condition": "selected_test_relative_return >= 30 and selected_test_max_dd <= 15 for every evaluated fold",
            "reason": "Do not convert single-fold relative uplift into promotion or retained actor evidence.",
        },
        {
            "rule": "ppo_vs_bc_drawdown_drift",
            "condition": "stop or reject checkpoint when selected_vs_bc_drawdown_delta > 2pp unless the absolute fold gate passes",
            "reason": "Existing sparse-release PPO improved relative return on wf_2022 but pushed MaxDD far above the target.",
        },
        {
            "rule": "target_scope_before_training",
            "condition": "eval_overlap_rows == 0 and fold boundary metadata has one variant per target fold",
            "reason": "Purged/fold-local labels are mandatory before BC or PPO target reuse.",
        },
        {
            "rule": "joint_target_coverage",
            "condition": "target table must contain current positive rebound lift rows and downtrend/risk-control rows in the same fold-local contract",
            "reason": "Release-only and drawdown-only targets each failed as-is; cap10 handoff needs a joint objective.",
        },
        {
            "rule": "wf_2020_coverage_before_promotion",
            "condition": "next full train/replay must add non-leaking wf_2020 target coverage before promotion review",
            "reason": "Current sparse-release PPO screen only evaluated wf_2022 and wf_2023_2024.",
        },
    ]
)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "cap10_training_objective_handoff_v1",
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_objective", "checkpoint_selection"],
    "decision": "cap10_foldlocal_multiobjective_not_training_ready_as_is" if readiness_gate_count == 0 else "cap10_foldlocal_multiobjective_training_candidate_found",
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "ready_target_asset_count": readiness_gate_count,
    "best_diagnostic_asset": best_asset,
    "next_mechanism": "build_fold_local_joint_sparse_release_drawdown_targets_with_checkpoint_stop_rules",
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "source_handoff_decision": source_selections["cap10_training_objective_handoff_v1"].get("decision", ""),
    "source_handoff_risk_gap": source_selections["cap10_training_objective_handoff_v1"].get("risk_gap", {}),
    "selected_training_failures": selected_training_failures,
    "promotion_blocker": "No existing target asset is safe to reuse as-is for cap10 multi-objective training: current release/drawdown targets either lack joint coverage or already produced mixed/failed fold screens.",
    "acceptance_gate_for_next_run": {
        "fold_local_target_scope_required": True,
        "eval_overlap_rows_required": 0,
        "current_positive_rebound_lift_rows_required": True,
        "downtrend_rows_required": True,
        "checkpoint_overfit_stop_rules_required": True,
        "relative_return_min_pp": TARGET_RELATIVE_RETURN,
        "max_dd_max_pct": TARGET_MAX_DD,
        "walkforward_required": True,
        "fixed_split_required": True,
        "wf_2020_target_coverage_required_before_promotion": True,
        "no_runtime_promotion_from_target_readiness_only": True,
    },
    "external_sources": sorted({
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://doi.org/10.21314/JCF.2016.322",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678",
        "https://arxiv.org/abs/2006.04779",
        "https://arxiv.org/abs/1506.02188",
    }),
    "sidecars": [
        "target_asset_readiness.csv",
        "target_scope_audit.csv",
        "target_lift_audit.csv",
        "fold_boundary_audit.csv",
        "training_screen_summary.csv",
        "anti_overfit_stop_rules.csv",
        "summary.json",
        "selection.json",
    ],
}

summary = {
    **selection,
    "target_asset_readiness_rows": int(len(target_asset_readiness)),
    "training_screen_rows": int(len(training_screen_summary)),
    "overfit_stop_flags": int(training_screen_summary.get("overfit_stop_flag", pd.Series(dtype=bool)).fillna(False).sum()) if not training_screen_summary.empty else 0,
}

# Persist sidecars.
target_asset_readiness.to_csv(OUT_DIR / "target_asset_readiness.csv", index=False)
target_scope_audit.to_csv(OUT_DIR / "target_scope_audit.csv", index=False)
target_lift_audit.to_csv(OUT_DIR / "target_lift_audit.csv", index=False)
fold_boundary_audit.to_csv(OUT_DIR / "fold_boundary_audit.csv", index=False)
training_screen_summary.to_csv(OUT_DIR / "training_screen_summary.csv", index=False)
anti_overfit_stop_rules.to_csv(OUT_DIR / "anti_overfit_stop_rules.csv", index=False)
(OUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))

selection


{'experiment': 'cap10_foldlocal_multi_objective_readiness_v1',
 'created_at': '2026-05-25T22:44:12.250299+00:00',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'cap10_training_objective_handoff_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection',
  'training_objective',
  'checkpoint_selection'],
 'decision': 'cap10_foldlocal_multiobjective_not_training_ready_as_is',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'ready_target_asset_count': 0,
 'best_diagnostic_asset': 'sparse_rebound_release_target_revision_v1',
 'next_mechanism': 'build_fold_local_joint_sparse_release_drawdown_targets_with_checkpoint_stop_rules',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'source_handoff_decision': 'train_state_conditioned_multi_objective',
 'source_handoff_risk_gap': {'target_max_dd': 15.0,
  'observed_max_dd': 22.02,
  'max_dd_gap': 7.02,
  'target_relative_return': 30.0,
  'observed_min_rela

In [6]:

conclusion_md = f"""
## Conclusion

Decision: `{selection['decision']}`.

No runtime promotion and no training branch is opened from this card. The best diagnostic asset is `{selection['best_diagnostic_asset']}`, but no existing target table is safe to reuse as-is for cap10 multi-objective training.

Key evidence:
- Ready target assets: `{selection['ready_target_asset_count']}` of `{len(target_asset_readiness)}`.
- Sparse-release target revision fixed fold-local scope/lift, but its PPO screen was mixed; `wf_2022` selected PPO violated the drawdown target.
- Drawdown-control fold-local targets contain downtrend rows, but no rebound rows, and the follow-up PPO screen failed fold gates.
- The original fold-local joint target contains both state families, but later target-priority audit showed rebound lift was erased before the sparse-release semantic revision.

Next action:
Build a single fold-local joint target contract that combines sparse-release rebound lift and drawdown-control state rows under one fold calendar, then train only with the anti-overfit stop rules written to `anti_overfit_stop_rules.csv`.
"""
display(Markdown(conclusion_md))



## Conclusion

Decision: `cap10_foldlocal_multiobjective_not_training_ready_as_is`.

No runtime promotion and no training branch is opened from this card. The best diagnostic asset is `sparse_rebound_release_target_revision_v1`, but no existing target table is safe to reuse as-is for cap10 multi-objective training.

Key evidence:
- Ready target assets: `0` of `3`.
- Sparse-release target revision fixed fold-local scope/lift, but its PPO screen was mixed; `wf_2022` selected PPO violated the drawdown target.
- Drawdown-control fold-local targets contain downtrend rows, but no rebound rows, and the follow-up PPO screen failed fold gates.
- The original fold-local joint target contains both state families, but later target-priority audit showed rebound lift was erased before the sparse-release semantic revision.

Next action:
Build a single fold-local joint target contract that combines sparse-release rebound lift and drawdown-control state rows under one fold calendar, then train only with the anti-overfit stop rules written to `anti_overfit_stop_rules.csv`.
